# DPO 학습 — Qwen2.5-3B-Instruct + LoRA

PRD_LLM_선호학습.md §9.2 실행 노트북.

**전제**: `data/dpo_pairs.jsonl` 파일을 Google Drive 또는 로컬 → 업로드해 둠.

**환경**: Google Colab 무료 T4 (16 GB VRAM) 또는 그 이상.

**산출물**: `qwen25-3b-outlook-dpo/` LoRA 어댑터 (~100 MB). 로컬 `OUTLOOK_LOCAL_LLM_ADAPTER_PATH`로 지정해서 사용.

## 0. 의존성 설치 (Colab 첫 셀)

In [ ]:
!pip install -q "transformers>=4.45" "trl>=0.11" "peft>=0.13" "bitsandbytes>=0.43" "accelerate>=0.34" "datasets>=3.0"

## 1. 데이터 업로드

Google Drive 마운트 후 `data/dpo_pairs.jsonl` 위치를 잡거나, Colab에 직접 업로드.

아래는 Drive 마운트 예시. 직접 업로드면 `PAIRS_PATH`만 바꾸면 됨.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PAIRS_PATH = '/content/drive/MyDrive/quantum/dpo_pairs.jsonl'
OUTPUT_DIR = '/content/drive/MyDrive/quantum/qwen25-3b-outlook-dpo'

## 2. 데이터셋 로드

In [ ]:
import json
from datasets import Dataset

rows = []
with open(PAIRS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        rows.append({
            'prompt': record['prompt'],
            'chosen': record['chosen'],
            'rejected': record['rejected'],
        })

dataset = Dataset.from_list(rows)
print(f'pairs: {len(dataset)}')
print(dataset[0])

## 3. Base 모델 + LoRA 로드 (4-bit)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
)
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## 4. DPO 트레이너 설정 + 학습

In [ ]:
from trl import DPOConfig, DPOTrainer

split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split['train']
eval_ds = split['test']

config = DPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    beta=0.1,
    max_length=2048,
    max_prompt_length=1600,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,
    bf16=True,
    report_to='none',
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

trainer.train()

## 5. LoRA 어댑터 저장 (Drive)

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('saved to:', OUTPUT_DIR)

## 6. 검증 — 한 샘플 생성

In [ ]:
sample = eval_ds[0]
messages = [{'role': 'user', 'content': sample['prompt']}]
chat_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(chat_input, return_tensors='pt').to(model.device)
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id,
    )
print(tokenizer.decode(output_ids[0, inputs['input_ids'].shape[-1]:], skip_special_tokens=True))
print('---')
print('chosen ref:', sample['chosen'])

## 7. 로컬로 가져오기

Drive에 저장된 `qwen25-3b-outlook-dpo/` 폴더를 로컬 Mac으로 복사.

```bash
# 로컬 Mac에서
export OUTLOOK_LOCAL_LLM_ADAPTER_PATH=/path/to/qwen25-3b-outlook-dpo
uvicorn web.main:app --reload
```

`OUTLOOK_LOCAL_LLM_ADAPTER_PATH`가 설정되고 `transformers`/`peft`/`torch`가 설치되어 있으면 `OutlookService`가 OpenAI 대신 학습된 Qwen 어댑터로 평가한다. 실패 시 자동으로 OpenAI 경로로 fallback.